# 02. 전처리 및 정밀 EDA (Preprocessing & Refined EDA)

- **작성자**: 김정현
- **목적**: 결측치·이상치 도메인 인과성 기반 정제, 전처리 파이프라인 검증 및 팀원 공유용 EDA 시각화
- **관련 모듈**: `src/preprocessing.py`, `src/viz.py`

> **[팀원 참고사항]**
> 본 노트북에 포함된 시각화는 **[전체 6,168행 전체 조망(Global Overview)]**과 **[결측치·이상치 발생 구간 10~36시간 국소 확대(Local Zoom-in)]**로 명확히 구분되어 있습니다.

---

## 1. 환경 설정 및 전처리 파이프라인 로드

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트 경로 추가
ROOT_DIR = Path.cwd().resolve().parent.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

from src.preprocessing import run_preprocessing_pipeline, load_raw_data
from src.viz import set_korean_font
from src.config import FIGURES_DIR, REPORTS_DIR

set_korean_font()

## 2. 전체 데이터 조망 (Global Timeseries Overview)
- **시각화 범위**: **2021년 1월 1일 ~ 9월 14일 전체 6,168시간 (전체 데이터 100%)**
- 전체 전력 사용량(상단) 및 생산량(하단) 시계열 트렌드와 함께, **결측치 및 이상치가 발생한 5개 지점(①~⑤)**의 전체적 위치를 조망합니다.

In [ ]:
fig0_path = FIGURES_DIR / "00_full_timeseries_overview.png"
if fig0_path.exists():
    display(Image(filename=str(fig0_path)))

## 3. 전처리 파이프라인 실행 및 전후 비교

In [ ]:
# 1. 원본 데이터 로드
df_raw = load_raw_data()
print(f"[원본 데이터] 크기: {df_raw.shape}, 결측치 총합: {df_raw.isnull().sum().sum()}개")

# 2. 전처리 파이프라인 원스톱 실행
df_clean = run_preprocessing_pipeline()
print(f"\n[정제 데이터] 크기: {df_clean.shape}, 결측치 총합: {df_clean.isnull().sum().sum()}개")

# 상위 5개 행 미리보기
df_clean.head()

## 4. 특이사항별 국소 확대 검증 (Local Zoom-in Plots)

### (1) 공장인원 결측 (17건) 및 셧다운 구간
- **시각화 범위**: **2021-08-28 09시 ~ 08-29 20시 (약 36시간 국소 확대)**
- **확대 사유**: 전체 6,168시간 중 17시간은 너무 좁아 전력/생산량과의 인과관계를 볼 수 없으므로 해당 구간을 돋보기처럼 확대한 플롯.
- **근거**: 붉은 음영 구간 동안 전력 0 kW, 생산량 0인 완전 셧다운 상태이므로 인원은 0.0명이 타당함.

In [ ]:
fig1_path = FIGURES_DIR / "01_missing_workers_shutdown.png"
if fig1_path.exists():
    display(Image(filename=str(fig1_path)))

### (2) 기상 풍속(3건) 및 강수량(1건) 결측치
- **시각화 범위**: **결측 발생 시점 전후 10~15시간 국소 확대**
- **확대 사유**: 1~2개 행 누락 직전/직후의 바람과 강수량 연속성을 확인하여 선형 보간의 타당성을 입증하기 위함.

In [ ]:
fig2_path = FIGURES_DIR / "02_missing_weather_cases.png"
if fig2_path.exists():
    display(Image(filename=str(fig2_path)))

### (3) 시간 컬럼 비정상 값(70~188) 복원 검증
- **시각화 범위**: **7/13 및 7/15 전후 각각 3일간의 24시간 기온 패턴 비교**
- **확대 사유**: 24개 행 순서대로 0~23시를 부여했을 때 전일/익일 기온 일변화 곡선(새벽 4시 최저, 오후 13~14시 최고)과 완벽 일치함을 입증하기 위함.

In [ ]:
fig3_path = FIGURES_DIR / "03_time_anomaly_restoration_validation.png"
if fig3_path.exists():
    display(Image(filename=str(fig3_path)))

### (4) 생산량 vs 인원 및 대기전력(Base Load) 분포
- **시각화 범위**: **전체 6,168개 행 전체 데이터 (Global Dataset)**
- **설명**: 생산량과 인원의 뚜렷한 선형 비례 관계 및 비가동 시에도 존재하는 대기전력(평균 46.5 kW)을 전체 데이터 분포로 확인.

In [ ]:
fig4_path = FIGURES_DIR / "04_production_power_relationship.png"
if fig4_path.exists():
    display(Image(filename=str(fig4_path)))

## 5. 정제 데이터 기초 통계 및 무결성 최종 검증

In [ ]:
print("=== 정제 완료 데이터 컬럼별 결측치 현황 ===")
print(df_clean.isnull().sum())

print("\n=== 수치형 컬럼 기술 통계 요약 ===")
display(df_clean.describe().round(2))

# 요약 통계량 CSV 파일 저장
clean_summary_path = REPORTS_DIR / "cleaned_data_summary.csv"
df_clean.describe().round(4).to_csv(clean_summary_path, encoding="utf-8-sig")
print(f"\n[저장 완료] 정제 데이터 통계 요약: {clean_summary_path}")